In [1]:
import pandas as pd
import numpy as np

# Display settings — show all columns, 2 decimal places for floats
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
eway_df = pd.read_csv("data_suspicious.csv")
eway_df = eway_df.drop(columns=["Unnamed: 0"], errors="ignore")
eway_df["ewb_dt"] = pd.to_datetime(eway_df["ewb_dt"], format="mixed")
eway_df["ewb_final_valid_dt"] = pd.to_datetime(eway_df["ewb_final_valid_dt"], format="mixed")
eway_df["time_gap"] = (eway_df.groupby("vehicle_number")["ewb_dt"].diff().dt.total_seconds()/ 3600)
print(eway_df.dtypes)
print(eway_df.head())

ewb_no                         int64
ewb_dt                datetime64[ns]
to_pin                         int64
travel_distance              float64
from_pin                       int64
ewb_final_valid_dt    datetime64[ns]
ewb_ass_amt                  float64
cgst_amt                     float64
sgst_amt                     float64
igst_amt                     float64
vehicle_number                object
time_gap                     float64
dtype: object
         ewb_no              ewb_dt  to_pin  travel_distance  from_pin  \
0  771456137920 2024-09-05 11:12:00  175125          1088.00    313324   
1  691777400326 2024-09-05 18:43:00  444105          1007.00    363642   
2  641777301769 2024-09-05 16:58:00  410208           419.00    390010   
3  611777859582 2024-09-06 17:15:00  151301          1287.00    390010   
4  641777318024 2024-09-05 17:15:00  521228          1382.00    390010   

   ewb_final_valid_dt  ewb_ass_amt  cgst_amt  sgst_amt  igst_amt  \
0 2024-09-11 23:59:59    2577

In [3]:
fasttag_df = pd.read_csv("fasttag_data_intern.csv")
fasttag_df["updated_at_npci"] = pd.to_datetime(fasttag_df["updated_at_npci"],format="%d/%m/%Y %H:%M:%S",errors="coerce")
fasttag_df["readertme"] = pd.to_datetime(fasttag_df["readertme"],format="%d/%m/%Y %I:%M:%S %p",errors="coerce")
fasttag_df.rename(columns={"readertme": "reader_time", "veh": "vehicle_number"}, inplace=True)
print(fasttag_df.dtypes)
print(fasttag_df.head())

toll_id                     int64
toll_name                  object
highway_type               object
geo_lat                   float64
geo_long                  float64
updated_at_npci    datetime64[ns]
status                     object
toll                      float64
reader_time        datetime64[ns]
vehicle_number             object
dtype: object
   toll_id                 toll_name highway_type  geo_lat  geo_long  \
0   103001  Jat Gangaicha Toll Plaza          Nat    28.28     76.61   
1   103001  Jat Gangaicha Toll Plaza          Nat    28.28     76.61   
2   103001  Jat Gangaicha Toll Plaza          Nat    28.28     76.61   
3   103001  Jat Gangaicha Toll Plaza          Nat    28.28     76.61   
4   103001  Jat Gangaicha Toll Plaza          Nat    28.28     76.61   

      updated_at_npci status      toll         reader_time vehicle_number  
0 2021-05-05 11:53:58      A 103001.00 2022-09-29 07:47:38     RJ14GP0632  
1 2021-05-05 11:53:58      A 103001.00 2022-05-20 09:26:28   

In [4]:
pincode_df = pd.read_csv("pincode_registry.csv",na_values=["NA", "N/A", ""])
pincode_df["pincode"] = pd.to_numeric(pincode_df["pincode"],errors="coerce").astype("Int64")
pincode_df["latitude"] = pd.to_numeric(pincode_df["latitude"],errors="coerce")
pincode_df["longitude"] = pd.to_numeric(pincode_df["longitude"],errors="coerce")
text_columns = ["circlename","regionname","divisionname","officename","officetype","delivery","district","statename"]
for col in text_columns:
    pincode_df[col] = pincode_df[col].str.strip()
print(pincode_df.dtypes)
print(pincode_df.head())
print(pincode_df[["latitude", "longitude"]].isna().sum())

circlename       object
regionname       object
divisionname     object
officename       object
pincode           Int64
officetype       object
delivery         object
district         object
statename        object
latitude        float64
longitude       float64
dtype: object
         circlename        regionname       divisionname    officename  \
0  Telangana Circle  Hyderabad Region  Adilabad Division  Kothimir B.O   
1  Telangana Circle  Hyderabad Region  Adilabad Division  Papanpet B.O   
2  Telangana Circle  Hyderabad Region  Adilabad Division    Kukuda B.O   
3  Telangana Circle  Hyderabad Region  Adilabad Division  Bareguda B.O   
4  Telangana Circle  Hyderabad Region  Adilabad Division     Mosam B.O   

   pincode officetype  delivery                district  statename  latitude  \
0   504273         BO  Delivery  KUMURAM BHEEM ASIFABAD  TELANGANA     19.36   
1   504299         BO  Delivery  KUMURAM BHEEM ASIFABAD  TELANGANA     19.48   
2   504299         BO  Delivery  KUMU

In [5]:
print("E-Way rows with missing values:", eway_df.isna().any(axis=1).sum())
print("FASTag rows with missing values:", fasttag_df.isna().any(axis=1).sum())
print("Pincode rows with missing values:", pincode_df.isna().any(axis=1).sum())
eway_df.dropna(inplace=True)
fasttag_df.dropna(inplace=True)
pincode_df.dropna(inplace=True)
eway_df.reset_index(drop=True, inplace=True)
fasttag_df.reset_index(drop=True, inplace=True)
pincode_df.reset_index(drop=True, inplace=True)

print("Duplicate E-Way Bills:", eway_df["ewb_no"].duplicated().sum())
eway_df.drop_duplicates(subset="ewb_no", keep="first",inplace=True)
eway_df.reset_index(drop=True, inplace=True)

E-Way rows with missing values: 42626
FASTag rows with missing values: 0
Pincode rows with missing values: 12632
Duplicate E-Way Bills: 1129


In [6]:
eway_df.head()

,ewb_no,ewb_dt,to_pin,travel_distance,from_pin,ewb_final_valid_dt,ewb_ass_amt,cgst_amt,sgst_amt,igst_amt,vehicle_number,time_gap
0,691777400326,2024-09-05 18:43:00,444105,1007.00,363642,2024-09-11 23:59:59,186662.00,0.00,0.00,33599.16,RJ42GA2022,7.52
1,611777859582,2024-09-06 17:15:00,151301,1287.00,390010,2024-09-13 23:59:59,94500.00,0.00,0.00,17010.00,GJ06BT8399,24.28
2,641777318024,2024-09-05 17:15:00,521228,1382.00,390010,2024-09-12 23:59:59,120000.00,0.00,0.00,0.00,GJ06BT8399,-24.00
3,601778158469,2024-09-07 23:53:00,635126,1436.00,390007,2024-09-15 23:59:59,196000.00,0.00,0.00,35280.00,GJ06BT8399,54.63
4,121933050480,2024-09-06 17:49:00,400063,704.00,500037,2024-09-10 23:59:59,12441540.00,0.00,0.00,2239477.20,RJ14GJ9674,-2.52


In [7]:
pincode_lookup = (pincode_df.groupby("pincode")[["latitude", "longitude"]].mean())
eway_df["from_lat"] = eway_df["from_pin"].map(pincode_lookup["latitude"])
eway_df["from_long"] = eway_df["from_pin"].map(pincode_lookup["longitude"])
eway_df["to_lat"] = eway_df["to_pin"].map(pincode_lookup["latitude"])
eway_df["to_long"] = eway_df["to_pin"].map(pincode_lookup["longitude"])
print(eway_df.head())

         ewb_no              ewb_dt  to_pin  travel_distance  from_pin  \
0  691777400326 2024-09-05 18:43:00  444105          1007.00    363642   
1  611777859582 2024-09-06 17:15:00  151301          1287.00    390010   
2  641777318024 2024-09-05 17:15:00  521228          1382.00    390010   
3  601778158469 2024-09-07 23:53:00  635126          1436.00    390007   
4  121933050480 2024-09-06 17:49:00  400063           704.00    500037   

   ewb_final_valid_dt  ewb_ass_amt  cgst_amt  sgst_amt   igst_amt  \
0 2024-09-11 23:59:59    186662.00      0.00      0.00   33599.16   
1 2024-09-13 23:59:59     94500.00      0.00      0.00   17010.00   
2 2024-09-12 23:59:59    120000.00      0.00      0.00       0.00   
3 2024-09-15 23:59:59    196000.00      0.00      0.00   35280.00   
4 2024-09-10 23:59:59  12441540.00      0.00      0.00 2239477.20   

  vehicle_number  time_gap  from_lat  from_long  to_lat  to_long  
0     RJ42GA2022      7.52     22.83      70.95   20.50    77.52  
1     

In [8]:
print("Missing FROM coordinates:")
print(eway_df[["from_lat", "from_long"]].isna().sum())
print("\nMissing TO coordinates:")
print(eway_df[["to_lat", "to_long"]].isna().sum())

# Removing
eway_df = eway_df.dropna(subset=["from_lat", "from_long", "to_lat", "to_long"]).reset_index(drop=True)
print("Remaining E-Way Bills:", len(eway_df))

Missing FROM coordinates:
from_lat     1508
from_long    1508
dtype: int64

Missing TO coordinates:
to_lat     889
to_long    889
dtype: int64
Remaining E-Way Bills: 153856


In [9]:
fault_df = pd.DataFrame({
    "ewb_no": eway_df["ewb_no"].to_numpy(),
    "expired_movement": False,
    "impossible_speed": False,
    "route_deviation": False,
    "toll_mismatch": False
})

In [10]:
# ==========================================
# CREATE FAULT DATAFRAME
# ==========================================
fault_df = pd.DataFrame({
    "ewb_no": eway_df["ewb_no"].to_numpy(),
    "expired_movement": False,
    "impossible_speed": False,
    "route_deviation": False,
    "toll_mismatch": False,
    "minimum_monetary_value": False,
})

In [11]:
# ==========================================
# PREPARE E-WAY DATA
# ==========================================
eway_expiry = eway_df[["ewb_no", "vehicle_number", "ewb_final_valid_dt"]].copy()
eway_expiry.dropna(subset=["vehicle_number", "ewb_final_valid_dt"], inplace=True)

# ==========================================
# PREPARE FASTAG DATA
# ==========================================
fasttag_movement = fasttag_df[["vehicle_number", "reader_time"]].copy()
fasttag_movement.dropna(subset=["vehicle_number", "reader_time"], inplace=True)

# ==========================================
# SORT DATA FOR MERGE_ASOF
# ==========================================
eway_expiry.sort_values("ewb_final_valid_dt", inplace=True)
fasttag_movement.sort_values("reader_time", inplace=True)

# ==========================================
# FIND FIRST FASTAG MOVEMENT AFTER EWB EXPIRY
# ==========================================
expired_check = pd.merge_asof(
    eway_expiry,
    fasttag_movement,
    left_on="ewb_final_valid_dt",
    right_on="reader_time",
    by="vehicle_number",
    direction="forward",
    allow_exact_matches=False
)

# ==========================================
# CHECK MOVEMENT WITHIN 24 HOURS OF EXPIRY
# ==========================================
expired_check["expired_movement"] = (
    expired_check["reader_time"].notna() &
    (expired_check["reader_time"] <= expired_check["ewb_final_valid_dt"] + pd.Timedelta(hours=24))
)

# ==========================================
# MAP RESULT TO FAULT DATAFRAME
# ==========================================
expired_map = expired_check.set_index("ewb_no")["expired_movement"]
fault_df["expired_movement"] = (
    fault_df["ewb_no"]
    .map(expired_map)
    .fillna(False)
    .astype(bool)
)

# ==========================================
# RESULTS
# ==========================================
print(fault_df.head())
print("\nExpired movement results:")
print(fault_df["expired_movement"].value_counts())
print("\nTotal fault rows:", len(fault_df))

         ewb_no  expired_movement  impossible_speed  route_deviation  \
0  691777400326             False             False            False   
1  611777859582             False             False            False   
2  641777318024             False             False            False   
3  601778158469             False             False            False   
4  121933050480             False             False            False   

   toll_mismatch  minimum_monetary_value  
0          False                   False  
1          False                   False  
2          False                   False  
3          False                   False  
4          False                   False  

Expired movement results:
expired_movement
False    153233
True        623
Name: count, dtype: int64

Total fault rows: 153856


In [12]:
# ============================================================
# RULE 2
#
# CONDITIONS:
# 1. SAME VEHICLE
# 2. EWB VALIDITY PERIODS OVERLAP
# 3. EACH EWB TRAVEL DISTANCE > 200 KM
# 4. DISTANCE BETWEEN EWB PAIR > 500 KM
#
# DISTANCE BETWEEN PAIR:
# EARLIER EWB DESTINATION -> LATER EWB ORIGIN
# ============================================================

# ============================================================
# PREPARE REQUIRED DATA
# ============================================================
rule2_df = eway_df[
    [
        "ewb_no", "vehicle_number", "ewb_dt", "ewb_final_valid_dt", 
        "travel_distance", "from_lat", "from_long", "to_lat", "to_long"
    ]
].dropna().copy()
# KEEP EWBs WITH TRAVEL DISTANCE > 200 KM
rule2_df = rule2_df[rule2_df["travel_distance"] > 200].copy()
# SORT BY VEHICLE AND EWB START TIME
rule2_df.sort_values(["vehicle_number", "ewb_dt"], inplace=True)
rule2_df.reset_index(drop=True, inplace=True)

print("EWBs eligible for Rule 2:", len(rule2_df))


# ============================================================
# VECTORIZED HAVERSINE DISTANCE FUNCTION
# ============================================================
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2.0) ** 2 +
        np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    a = np.clip(a, 0, 1)
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))

# ============================================================
# FIND ALL QUALIFIED OVERLAPPING PAIRS
# ============================================================
flagged_ewbs = set()
qualified_pair_count = 0
for vehicle_number, vehicle_df in rule2_df.groupby("vehicle_number", sort=False):
    # AT LEAST TWO EWBs REQUIRED
    if len(vehicle_df) < 2:
        continue
    # CONVERT VEHICLE DATA TO NUMPY ARRAYS
    ewb_no = vehicle_df["ewb_no"].to_numpy()
    ewb_dt = vehicle_df["ewb_dt"].to_numpy(dtype="datetime64[ns]")
    valid_dt = vehicle_df["ewb_final_valid_dt"].to_numpy(dtype="datetime64[ns]")
    
    from_lat = vehicle_df["from_lat"].to_numpy(dtype=np.float64)
    from_long = vehicle_df["from_long"].to_numpy(dtype=np.float64)
    to_lat = vehicle_df["to_lat"].to_numpy(dtype=np.float64)
    to_long = vehicle_df["to_long"].to_numpy(dtype=np.float64)
    total_ewbs = len(ewb_no)

    # ========================================================
    # CHECK EACH EWB AGAINST LATER EWBs
    # ========================================================
    for i in range(total_ewbs - 1):
        # DATA IS SORTED BY EWB_DT
        # FIND ALL LATER EWBs WHOSE START TIME IS BEFORE CURRENT EWB EXPIRES
        end_idx = np.searchsorted(ewb_dt, valid_dt[i], side="right")
        # NO LATER OVERLAPPING EWB
        if end_idx <= i + 1:
            continue
        candidate_idx = np.arange(i + 1, end_idx)
        # FULL INTERVAL OVERLAP CHECK
        overlap_mask = (valid_dt[candidate_idx] >= ewb_dt[i])
        candidate_idx = candidate_idx[overlap_mask]
        if candidate_idx.size == 0:
            continue
        # ====================================================
        # CALCULATE DISTANCE BETWEEN EWB PAIRS
        # CURRENT EWB DESTINATION -> LATER EWB ORIGIN
        # ====================================================
        distances = haversine_km(
            to_lat[i], to_long[i],
            from_lat[candidate_idx], from_long[candidate_idx]
        )
        # ====================================================
        # KEEP PAIRS > 500 KM APART
        # ====================================================
        qualified_idx = candidate_idx[distances > 500]
        if qualified_idx.size == 0:
            continue
        # ====================================================
        # FLAG BOTH EWBs
        # ====================================================
        flagged_ewbs.add(ewb_no[i])
        flagged_ewbs.update(ewb_no[qualified_idx].tolist())
        qualified_pair_count += qualified_idx.size


# ============================================================
# ADD RULE 2 RESULT TO FAULT_DF
# ============================================================
fault_df["distance_threshold"] = fault_df["ewb_no"].isin(flagged_ewbs)

# ============================================================
# PRINT FINAL RESULTS
# ============================================================
print("\n================================")
print("RULE 2 RESULTS")
print("================================")
print("Qualified EWB pairs:", qualified_pair_count)
print("EWBs flagged:", len(flagged_ewbs))

print("\nDistance threshold results:")
print(fault_df["distance_threshold"].value_counts())
print("\nFAULT_DF:")
print(fault_df.head())

EWBs eligible for Rule 2: 153856

RULE 2 RESULTS
Qualified EWB pairs: 2020176
EWBs flagged: 92355

Distance threshold results:
distance_threshold
True     92355
False    61501
Name: count, dtype: int64

FAULT_DF:
         ewb_no  expired_movement  impossible_speed  route_deviation  \
0  691777400326             False             False            False   
1  611777859582             False             False            False   
2  641777318024             False             False            False   
3  601778158469             False             False            False   
4  121933050480             False             False            False   

   toll_mismatch  minimum_monetary_value  distance_threshold  
0          False                   False               False  
1          False                   False                True  
2          False                   False                True  
3          False                   False                True  
4          False                   Fals

In [13]:
# ============================================================
# RULE 3: MINIMUM MONETARY VALUE
#
# Total EWB Value =
# ewb_ass_amt
#
# Condition:
# Total EWB Value > Rs. 50,000
# ============================================================
# CALCULATE TOTAL EWB VALUE
eway_df["total_ewb_value"] = (
    eway_df["ewb_ass_amt"]
)
# ADD RULE 3 RESULT TO FAULT_DF
monetary_map = eway_df.set_index("ewb_no")["total_ewb_value"] > 50000
fault_df["minimum_monetary_value"] = (
    fault_df["ewb_no"]
    .map(monetary_map)
    .fillna(False)
    .astype(bool)
)
# ============================================================
# RESULTS
# ============================================================
print("\n================================")
print("RULE 3 RESULTS")
print("================================")
print("EWBs with total value > Rs. 50,000:", fault_df["minimum_monetary_value"].sum())
print("\nMinimum monetary value results:")
print(fault_df["minimum_monetary_value"].value_counts())
print("\nSample fault_df:")
print(fault_df.head())


RULE 3 RESULTS
EWBs with total value > Rs. 50,000: 145663

Minimum monetary value results:
minimum_monetary_value
True     145663
False      8193
Name: count, dtype: int64

Sample fault_df:
         ewb_no  expired_movement  impossible_speed  route_deviation  \
0  691777400326             False             False            False   
1  611777859582             False             False            False   
2  641777318024             False             False            False   
3  601778158469             False             False            False   
4  121933050480             False             False            False   

   toll_mismatch  minimum_monetary_value  distance_threshold  
0          False                    True               False  
1          False                    True                True  
2          False                    True                True  
3          False                    True                True  
4          False                    True               False 

In [14]:
# ============================================================
# RULE 4: TIME AND VEHICLE OVERLAP
#
# CONDITIONS:
# 1. SAME VEHICLE NUMBER
# 2. EWB VALIDITY PERIODS OVERLAP
# 3. OVERLAP > 60%
#
# OVERLAP PERCENTAGE:
#
# overlap duration
# -----------------------------  x 100
# shorter EWB validity duration
#
# ============================================================

# ============================================================
# PREPARE DATA
# ============================================================
rule4_df = eway_df[["ewb_no", "vehicle_number", "ewb_dt", "ewb_final_valid_dt"]].dropna().copy()
# REMOVE INVALID VALIDITY PERIODS
rule4_df = rule4_df[rule4_df["ewb_final_valid_dt"] > rule4_df["ewb_dt"]].copy()
# SORT BY VEHICLE AND EWB START TIME
rule4_df.sort_values(["vehicle_number", "ewb_dt"], inplace=True)
rule4_df.reset_index(drop=True, inplace=True)

print("EWBs eligible for Rule 4:", len(rule4_df))


# ============================================================
# STORE FLAGGED EWB NUMBERS
# ============================================================
flagged_overlap_ewbs = set()
qualified_overlap_pair_count = 0


# ============================================================
# PROCESS VEHICLE BY VEHICLE
# ============================================================
for vehicle_number, vehicle_df in rule4_df.groupby("vehicle_number", sort=False):
    # AT LEAST TWO EWBs ARE REQUIRED
    if len(vehicle_df) < 2:
        continue
    # ========================================================
    # CONVERT TO NUMPY ARRAYS
    # ========================================================
    ewb_no = vehicle_df["ewb_no"].to_numpy()
    start_time = vehicle_df["ewb_dt"].to_numpy(dtype="datetime64[ns]")
    end_time = vehicle_df["ewb_final_valid_dt"].to_numpy(dtype="datetime64[ns]")
    total_ewbs = len(ewb_no)

    # ========================================================
    # CALCULATE EWB VALIDITY DURATION
    # ========================================================
    validity_duration = (end_time - start_time).astype("timedelta64[ns]").astype(np.int64)

    # ========================================================
    # CHECK EACH EWB AGAINST LATER EWBs
    # ========================================================
    for i in range(total_ewbs - 1):
        # ====================================================
        # FIND EWBs STARTING BEFORE CURRENT EWB EXPIRES
        # ====================================================
        end_idx = np.searchsorted(start_time, end_time[i], side="right")

        # NO POSSIBLE OVERLAPPING EWB
        if end_idx <= i + 1:
            continue
        candidate_idx = np.arange(i + 1, end_idx)
        # ====================================================
        # CALCULATE OVERLAP START AND END
        # ====================================================
        overlap_start = np.maximum(start_time[i], start_time[candidate_idx])
        overlap_end = np.minimum(end_time[i], end_time[candidate_idx])

        # ====================================================
        # CALCULATE OVERLAP DURATION
        # ====================================================
        overlap_duration = (overlap_end - overlap_start).astype("timedelta64[ns]").astype(np.int64)
        # REMOVE NON-OVERLAPPING PAIRS
        overlap_mask = (overlap_duration > 0)
        if not np.any(overlap_mask):
            continue
        candidate_idx = candidate_idx[overlap_mask]
        overlap_duration = overlap_duration[overlap_mask]
        # ====================================================
        # GET SHORTEST VALIDITY DURATION
        # ====================================================
        shortest_duration = np.minimum(validity_duration[i], validity_duration[candidate_idx])
        # ====================================================
        # CALCULATE OVERLAP RATIO
        # ====================================================
        overlap_ratio = overlap_duration / shortest_duration
        # ====================================================
        # KEEP OVERLAP > 60%
        # ====================================================
        qualified_mask = (overlap_ratio > 0.60)
        if not np.any(qualified_mask):
            continue
        qualified_idx = candidate_idx[qualified_mask]
        # ====================================================
        # FLAG BOTH EWBs
        # ====================================================
        flagged_overlap_ewbs.add(ewb_no[i])
        flagged_overlap_ewbs.update(ewb_no[qualified_idx].tolist())
        qualified_overlap_pair_count += qualified_idx.size

# ============================================================
# ADD RULE 4 RESULT TO FAULT_DF
# ============================================================
fault_df["time_vehicle_overlap"] = fault_df["ewb_no"].isin(flagged_overlap_ewbs)

# ============================================================
# PRINT RESULTS
# ============================================================
print("\n================================")
print("RULE 4 RESULTS")
print("================================")
print("Qualified overlapping EWB pairs:", qualified_overlap_pair_count)
print("EWBs flagged:", len(flagged_overlap_ewbs))
print("\nTime and vehicle overlap results:")
print(fault_df["time_vehicle_overlap"].value_counts())
print("\nFAULT_DF:")
print(fault_df.head())

EWBs eligible for Rule 4: 153856

RULE 4 RESULTS
Qualified overlapping EWB pairs: 2062409
EWBs flagged: 127617

Time and vehicle overlap results:
time_vehicle_overlap
True     127617
False     26239
Name: count, dtype: int64

FAULT_DF:
         ewb_no  expired_movement  impossible_speed  route_deviation  \
0  691777400326             False             False            False   
1  611777859582             False             False            False   
2  641777318024             False             False            False   
3  601778158469             False             False            False   
4  121933050480             False             False            False   

   toll_mismatch  minimum_monetary_value  distance_threshold  \
0          False                    True               False   
1          False                    True                True   
2          False                    True                True   
3          False                    True                True   
4          

In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# BEARING ANGLE DISCREPANCY
# PAIR-LEVEL THRESHOLD PREDICTION
# ============================================================

# ============================================================
# 1. CALCULATE BEARING
# ============================================================
def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1 = np.radians(np.asarray(lat1, dtype=np.float64))
    lon1 = np.radians(np.asarray(lon1, dtype=np.float64))
    lat2 = np.radians(np.asarray(lat2, dtype=np.float64))
    lon2 = np.radians(np.asarray(lon2, dtype=np.float64))

    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)

    bearing = np.degrees(np.arctan2(x, y))
    
    return (bearing + 360) % 360


# ============================================================
# 2. CALCULATE HAVERSINE DISTANCE
# ============================================================
def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(np.asarray(lat1, dtype=np.float64))
    lon1 = np.radians(np.asarray(lon1, dtype=np.float64))
    lat2 = np.radians(np.asarray(lat2, dtype=np.float64))
    lon2 = np.radians(np.asarray(lon2, dtype=np.float64))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    a = np.clip(a, 0, 1)

    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))


# ============================================================
# 3. CALCULATE BEARING FOR EVERY EWB
# ============================================================
eway_df["bearing_angle"] = calculate_bearing(
    eway_df["from_lat"], eway_df["from_long"],
    eway_df["to_lat"], eway_df["to_long"]
)

print("Bearing angles calculated:", len(eway_df))


# ============================================================
# 4. PREPARE EWB DATA
# ============================================================
bearing_df = eway_df[
    [
        "ewb_no", "vehicle_number", "ewb_dt", "ewb_final_valid_dt",
        "travel_distance", "ewb_ass_amt", "from_lat", "from_long",
        "to_lat", "to_long", "bearing_angle"
    ]
].dropna().copy()

bearing_df = bearing_df[bearing_df["ewb_final_valid_dt"] > bearing_df["ewb_dt"]].copy()

bearing_df.sort_values(["vehicle_number", "ewb_dt"], inplace=True)
bearing_df.reset_index(drop=True, inplace=True)

print("EWBs eligible for bearing analysis:", len(bearing_df))


# ============================================================
# 5. PAIR STORAGE
# ============================================================
pair_vehicle, pair_ewb_1, pair_ewb_2 = [], [], []
pair_bearing_1, pair_bearing_2, pair_divergence = [], [], []
pair_overlap_ratio, pair_distance = [], []
pair_over_200, pair_over_50000 = [], []


# ============================================================
# 6. FIND SAME-VEHICLE OVERLAPPING EWB PAIRS
# ============================================================
for vehicle_number, vehicle_df in bearing_df.groupby("vehicle_number", sort=False):

    if len(vehicle_df) < 2:
        continue

    ewb_no = vehicle_df["ewb_no"].to_numpy()
    start_time = vehicle_df["ewb_dt"].to_numpy(dtype="datetime64[ns]")
    end_time = vehicle_df["ewb_final_valid_dt"].to_numpy(dtype="datetime64[ns]")
    travel_distance = vehicle_df["travel_distance"].to_numpy(dtype=np.float64)
    ewb_value = vehicle_df["ewb_ass_amt"].to_numpy(dtype=np.float64)
    
    from_lat = vehicle_df["from_lat"].to_numpy(dtype=np.float64)
    from_long = vehicle_df["from_long"].to_numpy(dtype=np.float64)
    to_lat = vehicle_df["to_lat"].to_numpy(dtype=np.float64)
    to_long = vehicle_df["to_long"].to_numpy(dtype=np.float64)
    bearing = vehicle_df["bearing_angle"].to_numpy(dtype=np.float64)

    validity_duration = (end_time - start_time).astype("timedelta64[ns]").astype(np.int64)
    total_ewbs = len(ewb_no)

    # ========================================================
    # CHECK PAIRS
    # ========================================================
    for i in range(total_ewbs - 1):
        end_idx = np.searchsorted(start_time, end_time[i], side="right")

        if end_idx <= i + 1:
            continue

        candidate_idx = np.arange(i + 1, end_idx)

        # CALCULATE TIME OVERLAP
        overlap_start = np.maximum(start_time[i], start_time[candidate_idx])
        overlap_end = np.minimum(end_time[i], end_time[candidate_idx])
        overlap_duration = (overlap_end - overlap_start).astype("timedelta64[ns]").astype(np.int64)

        overlap_mask = (overlap_duration > 0)
        
        if not np.any(overlap_mask):
            continue

        candidate_idx = candidate_idx[overlap_mask]
        overlap_duration = overlap_duration[overlap_mask]

        # OVERLAP RATIO
        shortest_duration = np.minimum(validity_duration[i], validity_duration[candidate_idx])
        overlap_ratio = overlap_duration / shortest_duration

        # BEARING DIVERGENCE
        raw_difference = np.abs(bearing[i] - bearing[candidate_idx])
        divergence = np.minimum(raw_difference, 360 - raw_difference)

        # DISTANCE BETWEEN EWB PAIR (EWB 1 DESTINATION -> EWB 2 ORIGIN)
        pair_gap_distance = haversine_km(
            to_lat[i], to_long[i],
            from_lat[candidate_idx], from_long[candidate_idx]
        )

        # EACH EWB > 200 KM & EACH EWB > RS. 50,000
        each_over_200 = (travel_distance[i] > 200) & (travel_distance[candidate_idx] > 200)
        each_over_50000 = (ewb_value[i] > 50000) & (ewb_value[candidate_idx] > 50000)

        # STORE PAIRS
        pair_count = candidate_idx.size

        pair_vehicle.extend([vehicle_number] * pair_count)
        pair_ewb_1.extend([ewb_no[i]] * pair_count)
        pair_ewb_2.extend(ewb_no[candidate_idx].tolist())
        
        pair_bearing_1.extend([bearing[i]] * pair_count)
        pair_bearing_2.extend(bearing[candidate_idx].tolist())
        pair_divergence.extend(divergence.tolist())
        pair_overlap_ratio.extend(overlap_ratio.tolist())
        pair_distance.extend(pair_gap_distance.tolist())

        pair_over_200.extend(
            np.broadcast_to(each_over_200, pair_count).tolist()
            if np.ndim(each_over_200) == 0 else each_over_200.tolist()
        )

        pair_over_50000.extend(
            np.broadcast_to(each_over_50000, pair_count).tolist()
            if np.ndim(each_over_50000) == 0 else each_over_50000.tolist()
        )


# ============================================================
# 7. CREATE PAIR DATAFRAME
# ============================================================
df_overlaps = pd.DataFrame({
    "vehicle_number": pair_vehicle,
    "ewb_1": pair_ewb_1,
    "ewb_2": pair_ewb_2,
    "bearing_1": pair_bearing_1,
    "bearing_2": pair_bearing_2,
    "divergence_angle": pair_divergence,
    "overlap_ratio": pair_overlap_ratio,
    "distance_between_ewbs": pair_distance,
    "each_ewb_over_200km": pair_over_200,
    "each_ewb_over_50000": pair_over_50000
})


# ============================================================
# 8. PAIR-LEVEL BUSINESS CRITERIA
# ============================================================
df_overlaps["overlap_over_60"] = df_overlaps["overlap_ratio"] > 0.60
df_overlaps["distance_over_500"] = df_overlaps["distance_between_ewbs"] > 500


# ============================================================
# 9. CREATE SUSPICION PROXY LABEL (BEARING IS NOT USED HERE)
# ============================================================
df_overlaps["actual_suspicion"] = (
    df_overlaps["overlap_over_60"] &
    df_overlaps["each_ewb_over_200km"] &
    df_overlaps["distance_over_500"] &
    df_overlaps["each_ewb_over_50000"]
).astype(np.int8)


# ============================================================
# 10. PAIR RESULTS
# ============================================================
print("\n================================")
print("PAIR-LEVEL SUSPICION RESULTS")
print("================================")

print(df_overlaps["actual_suspicion"].value_counts())
print("\nTotal overlapping pairs:", len(df_overlaps))


# ============================================================
# 11. ANALYZE BEARING ANGLE RANGES
# ============================================================
print("\n================================")
print("BEARING ANGLE RANGE ANALYSIS")
print("================================")

# CREATE 5-DEGREE BEARING BINS
bearing_bins = np.arange(0, 185, 5)

df_overlaps["bearing_range"] = pd.cut(
    df_overlaps["divergence_angle"],
    bins=bearing_bins,
    include_lowest=True,
    right=False
)

# CALCULATE SUSPICION RATE FOR EACH RANGE
bearing_range_analysis = (
    df_overlaps.groupby("bearing_range", observed=True)
    .agg(
        total_pairs=("actual_suspicion", "size"),
        suspicious_pairs=("actual_suspicion", "sum")
    )
    .reset_index()
)

bearing_range_analysis["suspicion_rate"] = (
    bearing_range_analysis["suspicious_pairs"] / bearing_range_analysis["total_pairs"]
)
bearing_range_analysis["suspicion_percentage"] = bearing_range_analysis["suspicion_rate"] * 100

# CALCULATE BASELINE SUSPICION RATE
baseline_suspicion_rate = df_overlaps["actual_suspicion"].mean()

# CALCULATE LIFT
bearing_range_analysis["lift"] = bearing_range_analysis["suspicion_rate"] / baseline_suspicion_rate

# PRINT ALL RANGES
print("\nAll bearing ranges:")
print(bearing_range_analysis.to_string(index=False))
print("\nOverall baseline suspicion rate:", f"{baseline_suspicion_rate * 100:.2f}%")


# ============================================================
# FIND HIGH-RISK RANGES
# ============================================================
high_risk_ranges = (
    bearing_range_analysis[bearing_range_analysis["lift"] > 1]
    .sort_values("lift", ascending=False)
)

print("\n================================")
print("HIGH-RISK BEARING RANGES")
print("================================")
print(high_risk_ranges.to_string(index=False))


# ============================================================
# FIND BEST BEARING RANGE
# ============================================================
best_range = bearing_range_analysis.loc[bearing_range_analysis["lift"].idxmax()]

print("\n================================")
print("BEST BEARING RANGE")
print("================================")

print("Bearing range:", best_range["bearing_range"])
print("Total pairs:", int(best_range["total_pairs"]))
print("Suspicious pairs:", int(best_range["suspicious_pairs"]))
print("Suspicion percentage:", f"{best_range['suspicion_percentage']:.2f}%")
print("Lift:", f"{best_range['lift']:.4f}")

Bearing angles calculated: 153856
EWBs eligible for bearing analysis: 153856

PAIR-LEVEL SUSPICION RESULTS
actual_suspicion
0    1925288
1    1213145
Name: count, dtype: int64

Total overlapping pairs: 3138433

BEARING ANGLE RANGE ANALYSIS

All bearing ranges:
bearing_range  total_pairs  suspicious_pairs  suspicion_rate  suspicion_percentage  lift
       [0, 5)       791499            331310            0.42                 41.86  1.08
      [5, 10)       170787             52237            0.31                 30.59  0.79
     [10, 15)       117938             45687            0.39                 38.74  1.00
     [15, 20)        86702             34771            0.40                 40.10  1.04
     [20, 25)        83832             34972            0.42                 41.72  1.08
     [25, 30)        76248             33422            0.44                 43.83  1.13
     [30, 35)        77115             36530            0.47                 47.37  1.23
     [35, 40)        70151 

In [17]:
# ============================================================
# 12. APPLY DATA-DERIVED BEST BEARING RANGE
# ============================================================

BEARING_MIN = float(
    best_range["bearing_range"].left
)

BEARING_MAX = float(
    best_range["bearing_range"].right
)


# FLAG PAIRS INSIDE BEST BEARING RANGE

df_overlaps["bearing_angle_discrepancy"] = (
    (df_overlaps["divergence_angle"] >= BEARING_MIN)
    &
    (df_overlaps["divergence_angle"] < BEARING_MAX)
)


# ============================================================
# GET ALL EWBs INVOLVED IN FLAGGED PAIRS
# ============================================================

bearing_flagged_ewbs = pd.unique(
    pd.concat(
        [
            df_overlaps.loc[
                df_overlaps["bearing_angle_discrepancy"],
                "ewb_1"
            ],

            df_overlaps.loc[
                df_overlaps["bearing_angle_discrepancy"],
                "ewb_2"
            ]
        ],
        ignore_index=True
    )
)


# ============================================================
# ADD RESULT TO FAULT_DF
# ============================================================

fault_df["bearing_angle_discrepancy"] = (
    fault_df["ewb_no"]
    .isin(bearing_flagged_ewbs)
)


# ============================================================
# RESULTS
# ============================================================

print("\n================================")
print("FINAL BEARING CRITERION")
print("================================")

print(
    f"Data-derived bearing discrepancy range: "
    f"{BEARING_MIN:.0f}° <= difference < {BEARING_MAX:.0f}°"
)

print(
    "\nPairs inside bearing range:",
    df_overlaps["bearing_angle_discrepancy"].sum()
)

print(
    "\nBearing angle discrepancy results:"
)

print(
    fault_df["bearing_angle_discrepancy"]
    .value_counts()
)


FINAL BEARING CRITERION
Data-derived bearing discrepancy range: 30° <= difference < 35°

Pairs inside bearing range: 77115

Bearing angle discrepancy results:
bearing_angle_discrepancy
False    133704
True      20152
Name: count, dtype: int64
